# Ensemble Affinity Analysis (On-the-fly)

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [10]:
# Set run_id explicitly, or keep None to use latest run folder.
# run_id = "20260401_082538"
run_id = "20260406_211752"

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"

if not prp_path.exists() or not ra_path.exists():
    raise FileNotFoundError(
        "Merged files not found. Run scripts/run_postprocessing.py first for this run_id."
    )

prp_df = pd.read_csv(prp_path)
ra_df = pd.read_csv(ra_path)

ignore_models = ['smollm2-1.7b', 'gemma3-4b', 'phi3-3.8b', 'mistral-7b']
prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)]
ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)]
print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())

Run directory: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/20260406_211752
PRP rows: 1800 | Models: 6
RA rows: 1799 | Models: 6


In [11]:
# Quick coverage checks
display(prp_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())
display(ra_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())

,Model_Slug,n_abstracts
0,gemma4-31b-cloud,50
1,llama3.1-8b,50
2,llama4-16x17b,50
3,ministral-3-14b-cloud,50
4,mixtral-8x7b,50
5,qwen2.5-7b,50


,Model_Slug,n_abstracts
0,gemma4-31b-cloud,50
1,llama3.1-8b,50
2,llama4-16x17b,50
3,ministral-3-14b-cloud,50
4,mixtral-8x7b,50
5,qwen2.5-7b,50


# PRP Affinity level
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [12]:
fig = px.box(
    prp_df,
    x='Abstract_Index',
    y='LLM_Affinity',
    points='all',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution by Abstract'
 )
fig.update_xaxes(range=[5.5,11.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
fig.show()

In [13]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
# fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [14]:
# fig = px.box(
#     prp_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     points='all',
#     color = 'PRP_Name',
#     title=f'PRP Affinity Distribution by Abstract'
#  )
# fig.update_xaxes(range=[0.5,10.5])
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.show()

In [15]:
fig = px.box(
    prp_df[prp_df['Abstract_Index'].isin([1,8,20, 19])], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    # x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [16]:
fig = px.line(
    prp_df[prp_df['Abstract_Index'].isin([1,8,20, 19, 30])], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'PRP_Name',
    x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract',
    markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

General categories:
- 1 - Scenario Reduction Network Based on Wasserstein Distance With Regularization - Planning
- 8 - Simultaneous Assessment of Multiple Aspects of Stability of Power Systems With Renewable Generation -  Stability
- 19 - Chance-Constrained Pre-Contingency Joint Self- Scheduling of Energy and Reserve in VPP - DER, Services
- 20 - Dual Grid-Forming Control With Energy Regulation Capability of MMC-HVDC System Integrating Offshore Wind Farms and Weak Grids - IBR, Stability
- 30 - Synergistic Integration of Machine Learning and Mathematical Optimization for Unit Commitment - ????

Observations:
- Gemma: fails to identify planning for Index 1
- Gemma cloud: best classification
- Llama: miscelaneous performance
- Mistral: systematically assign too high affinities
- Phi & Smollm: systematically assign too high affinities
- Qwen: miscelaneous


## PRP Summaries

In [17]:
prp_summary = (
    prp_df.groupby(['Model_Slug', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(prp_summary.head(20))

,Model_Slug,PRP_Name,mean_affinity,std_affinity,n
0,gemma4-31b-cloud,CROF,37.00,25.852802,50
1,gemma4-31b-cloud,DER,37.76,32.764814,50
2,gemma4-31b-cloud,IBR,40.14,28.741643,50
3,gemma4-31b-cloud,Planning,53.40,27.019268,50
4,gemma4-31b-cloud,Stability,41.28,34.263707,50
5,gemma4-31b-cloud,System Services,45.10,26.039413,50
6,llama3.1-8b,CROF,18.40,13.606721,50
7,llama3.1-8b,DER,42.20,18.548337,50
8,llama3.1-8b,IBR,6.80,20.843440,50
9,llama3.1-8b,Planning,68.90,19.438285,50


In [18]:
# 
fig = px.scatter(
    prp_summary,
    color='PRP_Name',
    y='mean_affinity',
    x = 'Model_Slug',
    title='PRP Affinity Distribution by Model',
    # points = 'all',
 )
fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [19]:
prp_heat = prp_summary.pivot(index='Model_Slug', columns='PRP_Name', values='mean_affinity').sort_index()
fig = px.imshow(
    prp_heat,
    text_auto='.1f',
    color_continuous_scale='Viridis',
    title='Mean PRP Affinity (Model x PRP)',
    aspect='auto'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Model')
fig.show()

In [20]:
# Top-1 PRP agreement per abstract
prp_top1 = (
    prp_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'PRP_Name','LLM_Affinity']]
)

prp_agreement = (
    prp_top1.groupby('Abstract_Index')['PRP_Name']
    .nunique()
    .rename('n_unique_top1_prp')
    .reset_index()
)

print('Full agreement rate (all models same top-1 PRP):', (prp_agreement['n_unique_top1_prp'] == 1).mean())
display(prp_agreement['n_unique_top1_prp'].value_counts().sort_index().rename('n_abstracts'))

Full agreement rate (all models same top-1 PRP): 0.14


n_unique_top1_prp
1     7
2    15
3    18
4    10
Name: n_abstracts, dtype: int64

## RA Summaries

In [21]:
ra_df.head(20)

,Abstract_Index,Document Title,RA2025_ID,RA_Question,Cosine_x100,LLM_Affinity,Model_Slug,Run_ID
300,1,Scenario Reduction Network Based on Wasserstei...,37,what studies and metrics are required to ident...,NaN,20.0,gemma4-31b-cloud,20260406_211752
301,1,Scenario Reduction Network Based on Wasserstei...,38,how should grid enhancing technologies be incl...,NaN,10.0,gemma4-31b-cloud,20260406_211752
302,1,Scenario Reduction Network Based on Wasserstei...,39,what technical models need to be added to long...,NaN,20.0,gemma4-31b-cloud,20260406_211752
303,1,Scenario Reduction Network Based on Wasserstei...,40,what studies and metrics are required to evalu...,NaN,30.0,gemma4-31b-cloud,20260406_211752
304,1,Scenario Reduction Network Based on Wasserstei...,41,what additional probabilistic planning methods...,NaN,85.0,gemma4-31b-cloud,20260406_211752
305,1,Scenario Reduction Network Based on Wasserstei...,42,what additional planning models and methods ar...,NaN,80.0,gemma4-31b-cloud,20260406_211752
306,2,Domain of Stability Characterization for Power...,7,what approaches can be taken to track the dyna...,NaN,45.0,gemma4-31b-cloud,20260406_211752
307,2,Domain of Stability Characterization for Power...,8,"what methods, including data types, and analys...",NaN,40.0,gemma4-31b-cloud,20260406_211752
308,2,Domain of Stability Characterization for Power...,9,what analytical tools and models should be pro...,NaN,85.0,gemma4-31b-cloud,20260406_211752
309,2,Domain of Stability Characterization for Power...,10,what methods and tools should be used in plann...,NaN,30.0,gemma4-31b-cloud,20260406_211752


In [22]:
# fig = px.box(
#     ra_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     color='Model_Slug',
#     points='all',
#     # facet_row='PRP_Name',
#     # color = 'PRP_Name',
#     title=f'RA Affinity Distribution by Abstract'
#  )
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()

In [52]:
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df[ra_df['Abstract_Index'].isin([1,8,20,19])],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [24]:
ra_summary = (
    ra_df.groupby(['Model_Slug', 'RA2025_ID'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(ra_summary.head(20))

,Model_Slug,RA2025_ID,mean_affinity,std_affinity,n
0,gemma4-31b-cloud,1,41.875000,27.894892,8
1,gemma4-31b-cloud,2,28.750000,44.417527,4
2,gemma4-31b-cloud,3,60.000000,40.207794,4
3,gemma4-31b-cloud,4,63.750000,42.695628,4
4,gemma4-31b-cloud,5,45.000000,36.968455,4
5,gemma4-31b-cloud,6,11.250000,8.539126,4
6,gemma4-31b-cloud,7,63.888889,19.965248,9
7,gemma4-31b-cloud,8,40.555556,15.092309,9
8,gemma4-31b-cloud,9,79.444444,4.639804,9
9,gemma4-31b-cloud,10,33.333333,5.000000,9


In [46]:
fig = px.box(
    prp_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [25]:
fig = px.box(
    ra_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [26]:
# Top-1 RA agreement per abstract
ra_top1 = (
    ra_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'RA2025_ID']]
)

ra_agreement = (
    ra_top1.groupby('Abstract_Index')['RA2025_ID']
    .nunique()
    .rename('n_unique_top1_ra')
    .reset_index()
)

print('Full agreement rate (all models same top-1 RA):', (ra_agreement['n_unique_top1_ra'] == 1).mean())
display(ra_agreement['n_unique_top1_ra'].value_counts().sort_index().rename('n_abstracts'))

Full agreement rate (all models same top-1 RA): 0.0


n_unique_top1_ra
2     4
3     8
4    16
5    17
6     5
Name: n_abstracts, dtype: int64

## Coherency Analysis: Model Agreement on Affinity Levels

Analyze whether all models assign similar affinity scores to the same questions/abstracts.

## Coherency Logic: Coefficient of Variation (CV)

The coefficient of variation measures how dispersed the model affinities are relative to their average value.  
For each abstract-target unit, we collect the affinity scores from all models and compute:

$$
\mu = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\mu)^2}
$$

$$
CV = \frac{\sigma}{\mu + \varepsilon}
$$

where:

- $x_i$ is the affinity score assigned by model $i$
- $n$ is the number of models
- $\mu$ is the mean affinity
- $\sigma$ is the sample standard deviation
- $\varepsilon$ is a small constant added to avoid division by zero

Interpretation:

- Low CV means the models are in close agreement
- High CV means the models disagree more strongly
- CV can become unstable when the mean affinity is very small, so low-mean cases should be interpreted with caution

In [27]:
def affinity_level_fixed(score):
    if pd.isna(score):
        return pd.NA
    try:
        x = float(score)
    except Exception:
        return pd.NA
    if 0 <= x <= 40:
        return 'Low'
    if 40 < x <= 70:
        return 'Moderate'
    if 70 < x <= 100:
        return 'High'
    return pd.NA


def compute_unit_coherency(df, target_col, eps=1e-8, low_mean_threshold=5.0):
    required_cols = {'Abstract_Index', 'Model_Slug', target_col, 'LLM_Affinity'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    unit_stats = (
        df.groupby(['Abstract_Index', target_col], as_index=False)['LLM_Affinity']
        .agg(
            mean_affinity='mean',
            std_affinity='std',
            min_affinity='min',
            max_affinity='max',
            n_models='count',
            mad_affinity=lambda s: float(np.mean(np.abs(s - s.mean()))),
        )
    )
    unit_stats['cv'] = unit_stats['std_affinity'] / (unit_stats['mean_affinity'] + eps)
    unit_stats.loc[unit_stats['n_models'] < 2, 'cv'] = np.nan
    unit_stats['range'] = unit_stats['max_affinity'] - unit_stats['min_affinity']
    unit_stats['Affinity_Level'] = unit_stats['mean_affinity'].apply(affinity_level_fixed)
    unit_stats['cv_low_mean_flag'] = unit_stats['mean_affinity'] < float(low_mean_threshold)
    unit_stats['singleton_flag'] = unit_stats['n_models'] < 2
    return unit_stats


def summarize_coherency_by_level(unit_stats):
    level_order = pd.CategoricalDtype(categories=['Low', 'Moderate', 'High'], ordered=True)
    out = unit_stats.copy()
    out['Affinity_Level'] = out['Affinity_Level'].astype(level_order)

    level_summary = (
        out.dropna(subset=['Affinity_Level'])
        .groupby('Affinity_Level', as_index=False, observed=False)
        .agg(
            n_units=('Affinity_Level', 'count'),
            mean_cv=('cv', 'mean'),
            median_cv=('cv', 'median'),
            p90_cv=('cv', lambda s: float(np.percentile(s.dropna(), 90)) if s.notna().any() else np.nan),
            mean_mad=('mad_affinity', 'mean'),
            median_mad=('mad_affinity', 'median'),
            mean_range=('range', 'mean'),
            pct_low_mean_flag=('cv_low_mean_flag', 'mean'),
            pct_singleton=('singleton_flag', 'mean'),
        )
        .sort_values('Affinity_Level')
    )
    level_summary['pct_low_mean_flag'] = 100.0 * level_summary['pct_low_mean_flag']
    level_summary['pct_singleton'] = 100.0 * level_summary['pct_singleton']
    return level_summary


def compute_pairwise_disagreement_by_level(df, target_col):
    records = []
    for (abstract_idx, target_id), grp in df.groupby(['Abstract_Index', target_col]):
        grp2 = grp[['Model_Slug', 'LLM_Affinity']].dropna()
        if grp2['Model_Slug'].nunique() < 2:
            continue

        level = affinity_level_fixed(grp2['LLM_Affinity'].mean())
        rows = grp2.sort_values('Model_Slug')[['Model_Slug', 'LLM_Affinity']].to_records(index=False)

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                m1, a1 = rows[i]
                m2, a2 = rows[j]
                records.append(
                    {
                        'Abstract_Index': abstract_idx,
                        'Target_ID': target_id,
                        'Affinity_Level': level,
                        'Model_1': str(m1),
                        'Model_2': str(m2),
                        'Affinity_Difference': abs(float(a1) - float(a2)),
                    }
                )

    pairwise = pd.DataFrame(records)
    if pairwise.empty:
        return pairwise, pairwise

    pairwise_summary = (
        pairwise.dropna(subset=['Affinity_Level'])
        .groupby(['Affinity_Level', 'Model_1', 'Model_2'], as_index=False)
        .agg(mean_diff=('Affinity_Difference', 'mean'), median_diff=('Affinity_Difference', 'median'), std_diff=('Affinity_Difference', 'std'), n_pairs=('Affinity_Difference', 'count'))
        .sort_values(['Affinity_Level', 'mean_diff'], ascending=[True, False])
    )
    return pairwise, pairwise_summary

In [28]:
prp_coherency = compute_unit_coherency(prp_df, target_col='PRP_Name')

print("PRP Coherency Summary:")
print(f"Mean CV across all abstract-PRP pairs: {prp_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(prp_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-PRP pairs:")
display(prp_coherency.nlargest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-PRP pairs:")
display(prp_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])

PRP Coherency Summary:
Mean CV across all abstract-PRP pairs: 0.606
Abstracts with perfect agreement (CV=0): 0

Bottom 10 coherent (high CV) abstract-PRP pairs:


,Abstract_Index,PRP_Name,mean_affinity,cv,range
62,11,IBR,8.333333,1.594992,30.0
200,34,IBR,6.666667,1.549193,20.0
204,35,CROF,22.000000,1.369306,75.0
206,35,IBR,8.000000,1.369306,20.0
209,35,System Services,16.000000,1.351359,50.0
140,24,IBR,32.500000,1.252689,90.0
8,2,IBR,20.000000,1.224745,60.0
152,26,IBR,20.000000,1.224745,60.0
216,37,CROF,22.500000,1.223232,75.0
50,9,IBR,28.333333,1.215813,75.0



Top 10 coherent (low CV) abstract-PRP pairs:


,Abstract_Index,PRP_Name,mean_affinity,cv,range
76,13,Stability,89.166667,0.055132,15.0
166,28,Stability,89.166667,0.055132,15.0
112,19,Stability,84.166667,0.058407,10.0
10,2,Stability,89.166667,0.065554,15.0
262,44,Stability,86.666667,0.086858,20.0
46,8,Stability,88.333333,0.105390,25.0
232,39,Stability,82.833333,0.110755,22.0
116,20,IBR,85.000000,0.117647,25.0
118,20,Stability,70.000000,0.119523,20.0
103,18,DER,82.500000,0.152120,35.0


In [29]:
prp_coherency

,Abstract_Index,PRP_Name,mean_affinity,std_affinity,min_affinity,max_affinity,n_models,mad_affinity,cv,range,Affinity_Level,cv_low_mean_flag,singleton_flag
0,1,CROF,18.333333,16.020820,10.0,50.0,6,11.111111,0.873863,40.0,Low,False,False
1,1,DER,33.333333,15.055453,20.0,60.0,6,11.111111,0.451664,40.0,Low,False,False
2,1,IBR,16.666667,18.618987,0.0,40.0,6,16.666667,1.117139,40.0,Low,False,False
3,1,Planning,60.000000,28.284271,20.0,90.0,6,23.333333,0.471405,70.0,Moderate,False,False
4,1,Stability,41.666667,16.020820,20.0,60.0,6,12.222222,0.384500,40.0,Moderate,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,50,DER,64.166667,19.600170,40.0,90.0,6,15.555556,0.305457,50.0,Moderate,False,False
296,50,IBR,25.833333,27.643565,0.0,75.0,6,19.166667,1.070073,75.0,Low,False,False
297,50,Planning,35.833333,24.579802,5.0,70.0,6,18.888889,0.685948,65.0,Low,False,False
298,50,Stability,41.666667,20.412415,20.0,80.0,6,12.777778,0.489898,60.0,Moderate,False,False


In [51]:
fig1 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nlargest(7, 'cv')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (top CV abstracts)'
 )
fig2 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nsmallest(5, 'cv')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (low CV abstracts)'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig1.update_layout(yaxis_title='LLM Affinity')
fig2.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig1.show()
fig2.show()

In [31]:
ra_coherency = compute_unit_coherency(ra_df, target_col='RA2025_ID')

print("RA Coherency Summary:")
print(f"Mean CV across all abstract-RA pairs: {ra_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(ra_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-RA pairs:")
display(ra_coherency.nlargest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-RA pairs:")
display(ra_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])

RA Coherency Summary:
Mean CV across all abstract-RA pairs: 0.557
Abstracts with perfect agreement (CV=0): 14

Bottom 10 coherent (high CV) abstract-RA pairs:


,Abstract_Index,RA2025_ID,mean_affinity,cv,range
186,13,37,45.0,1.414214,90.0
746,48,10,37.5,1.414214,75.0
404,29,1,32.5,1.414214,65.0
743,48,7,22.5,1.414214,45.0
516,34,32,20.0,1.414214,40.0
517,34,33,20.0,1.414214,40.0
759,49,17,20.0,1.414214,40.0
745,48,9,17.5,1.414214,35.0
137,11,7,15.0,1.414214,30.0
138,11,8,15.0,1.414214,30.0



Top 10 coherent (low CV) abstract-RA pairs:


,Abstract_Index,RA2025_ID,mean_affinity,cv,range
40,4,14,30.0,0.0,0.0
42,4,16,30.0,0.0,0.0
46,4,30,80.0,0.0,0.0
63,5,29,40.0,0.0,0.0
99,7,38,10.0,0.0,0.0
167,12,28,45.0,0.0,0.0
271,18,38,60.0,0.0,0.0
333,23,15,30.0,0.0,0.0
341,23,33,60.0,0.0,0.0
547,36,9,80.0,0.0,0.0


In [32]:
# Abstract-level coherency: aggregate CV per abstract (for both PRP and RA)
prp_abstract_coherency = (
    prp_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

ra_abstract_coherency = (
    ra_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

print("PRP Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(prp_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(prp_abstract_coherency.tail(10))

print("\n" + "="*60)
print("\nRA Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(ra_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(ra_abstract_coherency.tail(10))

PRP Abstract-level Coherency (Average CV per Abstract):
Most coherent abstracts (lowest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
18,19,0.257569,0.375500,0.058407,0.124057,6
43,44,0.312951,0.539559,0.086858,0.166161,6
19,20,0.350472,0.657267,0.117647,0.210191,6
41,42,0.421874,0.864643,0.159415,0.254074,6
38,39,0.462652,0.721192,0.110755,0.207566,6
17,18,0.479590,0.787925,0.152120,0.205528,6
15,16,0.481308,0.585540,0.347009,0.093789,6
12,13,0.490364,0.784193,0.055132,0.245747,6
14,15,0.496641,0.883176,0.252982,0.208544,6
47,48,0.506184,0.737394,0.279682,0.169006,6



Least coherent abstracts (highest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
7,8,0.699751,0.938749,0.105390,0.326170,6
45,46,0.724339,1.106591,0.331413,0.273538,6
46,47,0.744046,0.894427,0.589915,0.129201,6
36,37,0.752047,1.223232,0.442133,0.339970,6
2,3,0.772938,1.179830,0.420758,0.257162,6
6,7,0.779826,1.173788,0.570875,0.227167,6
1,2,0.795687,1.224745,0.065554,0.385162,6
33,34,0.872350,1.549193,0.560686,0.362995,6
10,11,0.954565,1.594992,0.657994,0.335917,6
34,35,1.021237,1.369306,0.478429,0.402840,6




RA Abstract-level Coherency (Average CV per Abstract):
Most coherent abstracts (lowest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
16,17,0.224970,0.415945,0.030744,0.138158,6
0,1,0.232295,0.666667,0.062782,0.221711,6
11,12,0.258421,0.866025,0.000000,0.237058,12
18,19,0.259989,0.434923,0.121218,0.104368,7
49,50,0.303831,0.422353,0.176777,0.092643,6
24,25,0.312789,0.461838,0.181818,0.124524,6
3,4,0.358944,0.875466,0.000000,0.276273,12
30,31,0.372423,1.000000,0.091161,0.289882,12
37,38,0.374204,0.773640,0.000000,0.203276,12
9,10,0.389831,0.601962,0.074277,0.243502,6



Least coherent abstracts (highest mean CV):


,Abstract_Index,mean_cv,max_cv,min_cv,std_cv,n_pairs
29,30,0.679716,1.099944,0.346410,0.263518,12
39,40,0.691811,1.414214,0.128565,0.453381,11
2,3,0.733041,1.092599,0.405720,0.223706,6
47,48,0.814117,1.414214,0.162362,0.580309,11
15,16,0.825011,1.060660,0.471405,0.145860,12
36,37,0.891338,1.249000,0.654654,0.223583,6
48,49,0.895552,1.414214,0.000000,0.457702,12
28,29,0.971006,1.414214,0.108786,0.573846,12
33,34,0.972029,1.414214,0.101885,0.518418,12
10,11,1.414214,1.414214,1.414214,0.000000,6


In [33]:
# Visualization: Coherency Distribution
fig = px.histogram(
    prp_coherency[prp_coherency['cv'].notna()],
    x='cv',
    # nbins=30,
    # facet_col = 'PRP_Name',
    title='PRP Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=prp_coherency['cv'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean CV: {prp_coherency['cv'].mean():.3f}")
fig.show()

fig = px.histogram(
    ra_coherency[ra_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='RA Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=ra_coherency['cv'].mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean CV: {ra_coherency['cv'].mean():.3f}")
fig.show()

In [34]:
prp_coherency

,Abstract_Index,PRP_Name,mean_affinity,std_affinity,min_affinity,max_affinity,n_models,mad_affinity,cv,range,Affinity_Level,cv_low_mean_flag,singleton_flag
0,1,CROF,18.333333,16.020820,10.0,50.0,6,11.111111,0.873863,40.0,Low,False,False
1,1,DER,33.333333,15.055453,20.0,60.0,6,11.111111,0.451664,40.0,Low,False,False
2,1,IBR,16.666667,18.618987,0.0,40.0,6,16.666667,1.117139,40.0,Low,False,False
3,1,Planning,60.000000,28.284271,20.0,90.0,6,23.333333,0.471405,70.0,Moderate,False,False
4,1,Stability,41.666667,16.020820,20.0,60.0,6,12.222222,0.384500,40.0,Moderate,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,50,DER,64.166667,19.600170,40.0,90.0,6,15.555556,0.305457,50.0,Moderate,False,False
296,50,IBR,25.833333,27.643565,0.0,75.0,6,19.166667,1.070073,75.0,Low,False,False
297,50,Planning,35.833333,24.579802,5.0,70.0,6,18.888889,0.685948,65.0,Low,False,False
298,50,Stability,41.666667,20.412415,20.0,80.0,6,12.777778,0.489898,60.0,Moderate,False,False


In [47]:
# Abstract-level coherency scatter plots
fig = px.box(
    prp_coherency,
    x='Abstract_Index',
    y='cv',
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    points ='all',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()

fig = px.box(
    ra_coherency,
    x='Abstract_Index',
    y='cv',
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()


In [36]:
# Abstract-level coherency scatter plots
fig = px.scatter(
    prp_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

fig = px.scatter(
    ra_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

In [37]:
# Pairwise Model Coherency: Correlation between model pairs per abstract-PRP
# This shows which model pairs agree most on affinity levels

from itertools import combinations

# For PRP: compute correlation between each pair of models per abstract
prp_model_pairs = []
for abstract_idx in prp_df['Abstract_Index'].unique():
    for prp_name in prp_df[prp_df['Abstract_Index'] == abstract_idx]['PRP_Name'].unique():
        subset = prp_df[(prp_df['Abstract_Index'] == abstract_idx) & (prp_df['PRP_Name'] == prp_name)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    prp_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'PRP_Name': prp_name,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

prp_pairwise = pd.DataFrame(prp_model_pairs)

if len(prp_pairwise) > 0:
    print("PRP Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats.sort_values('mean_diff', ascending=True).head(10))

PRP Model Pair Agreement Analysis:

Model pairs with highest average disagreement (highest mean affinity difference):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
12,ministral-3-14b-cloud,mixtral-8x7b,27.840136,23.619097,294
4,gemma4-31b-cloud,qwen2.5-7b,26.966667,20.283910,300
8,llama3.1-8b,qwen2.5-7b,26.723333,23.093386,300
13,ministral-3-14b-cloud,qwen2.5-7b,26.390000,21.586674,300
6,llama3.1-8b,ministral-3-14b-cloud,26.000000,22.138965,300
3,gemma4-31b-cloud,mixtral-8x7b,25.234694,20.460968,294
11,llama4-16x17b,qwen2.5-7b,25.223333,18.207650,300
0,gemma4-31b-cloud,llama3.1-8b,24.963333,20.571321,300
14,mixtral-8x7b,qwen2.5-7b,22.897959,19.459851,294
1,gemma4-31b-cloud,llama4-16x17b,22.563333,15.939874,300



Model pairs with lowest average disagreement (highest mean agreement):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
5,llama3.1-8b,llama4-16x17b,19.266667,13.045927,300
9,llama4-16x17b,ministral-3-14b-cloud,20.000000,14.896522,300
2,gemma4-31b-cloud,ministral-3-14b-cloud,20.263333,17.826758,300
7,llama3.1-8b,mixtral-8x7b,20.578231,20.831823,294
10,llama4-16x17b,mixtral-8x7b,22.125850,14.522707,294
1,gemma4-31b-cloud,llama4-16x17b,22.563333,15.939874,300
14,mixtral-8x7b,qwen2.5-7b,22.897959,19.459851,294
0,gemma4-31b-cloud,llama3.1-8b,24.963333,20.571321,300
11,llama4-16x17b,qwen2.5-7b,25.223333,18.207650,300
3,gemma4-31b-cloud,mixtral-8x7b,25.234694,20.460968,294


In [38]:
# For RA: compute correlation between each pair of models per abstract
ra_model_pairs = []
for abstract_idx in ra_df['Abstract_Index'].unique():
    for ra_id in ra_df[ra_df['Abstract_Index'] == abstract_idx]['RA2025_ID'].unique():
        subset = ra_df[(ra_df['Abstract_Index'] == abstract_idx) & (ra_df['RA2025_ID'] == ra_id)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    ra_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'RA2025_ID': ra_id,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

ra_pairwise = pd.DataFrame(ra_model_pairs)

if len(ra_pairwise) > 0:
    print("RA Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats_ra.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats_ra.sort_values('mean_diff', ascending=True).head(10))

RA Model Pair Agreement Analysis:

Model pairs with highest average disagreement (highest mean affinity difference):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
7,llama3.1-8b,mixtral-8x7b,30.673077,18.258193,52
6,llama3.1-8b,ministral-3-14b-cloud,30.620155,20.039100,129
8,llama3.1-8b,qwen2.5-7b,26.564103,18.525931,78
13,ministral-3-14b-cloud,qwen2.5-7b,24.430657,17.993788,137
5,llama3.1-8b,llama4-16x17b,23.551136,14.862328,176
0,gemma4-31b-cloud,llama3.1-8b,22.391304,20.408398,138
12,ministral-3-14b-cloud,mixtral-8x7b,21.485507,13.581458,138
4,gemma4-31b-cloud,qwen2.5-7b,19.794118,15.552595,136
9,llama4-16x17b,ministral-3-14b-cloud,19.720000,16.017933,125
10,llama4-16x17b,mixtral-8x7b,16.917808,12.545200,73



Model pairs with lowest average disagreement (highest mean agreement):


,Model_1,Model_2,mean_diff,std_diff,n_pairs
14,mixtral-8x7b,qwen2.5-7b,12.351852,9.462484,108
1,gemma4-31b-cloud,llama4-16x17b,14.897959,14.111462,147
2,gemma4-31b-cloud,ministral-3-14b-cloud,15.728477,13.945099,151
3,gemma4-31b-cloud,mixtral-8x7b,16.238095,11.803295,105
11,llama4-16x17b,qwen2.5-7b,16.326316,11.349798,95
10,llama4-16x17b,mixtral-8x7b,16.917808,12.545200,73
9,llama4-16x17b,ministral-3-14b-cloud,19.720000,16.017933,125
4,gemma4-31b-cloud,qwen2.5-7b,19.794118,15.552595,136
12,ministral-3-14b-cloud,mixtral-8x7b,21.485507,13.581458,138
0,gemma4-31b-cloud,llama3.1-8b,22.391304,20.408398,138


In [39]:
# Model Agreement Heatmaps: Show which models have smallest disagreements
if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    # Create a symmetrical matrix for heatmap
    all_models = sorted(set(list(pair_stats['Model_1'].unique()) + list(pair_stats['Model_2'].unique())))
    model_agreement_matrix = pd.DataFrame(0.0, index=all_models, columns=all_models)
    
    for _, row in pair_stats.iterrows():
        model_agreement_matrix.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='PRP Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    all_models_ra = sorted(set(list(pair_stats_ra['Model_1'].unique()) + list(pair_stats_ra['Model_2'].unique())))
    model_agreement_matrix_ra = pd.DataFrame(0.0, index=all_models_ra, columns=all_models_ra)
    
    for _, row in pair_stats_ra.iterrows():
        model_agreement_matrix_ra.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix_ra.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix_ra,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='RA Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

## Coherency Key Insights

Summary of model agreement levels on affinity assignments.

In [40]:
# Summary statistics on model coherency
print("=" * 70)
print("COHERENCY ANALYSIS SUMMARY")
print("=" * 70)

print("\n📊 PRP COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {prp_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {prp_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(prp_coherency['cv'] == 0).sum()} / {len(prp_coherency)}")
print(f"  • High disagreement cases (CV>1): {(prp_coherency['cv'] > 1).sum()} / {len(prp_coherency)}")
print(f"  • Range of affinities (mean): {prp_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {prp_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {prp_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

print("\n📊 RA COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {ra_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {ra_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(ra_coherency['cv'] == 0).sum()} / {len(ra_coherency)}")
print(f"  • High disagreement cases (CV>1): {(ra_coherency['cv'] > 1).sum()} / {len(ra_coherency)}")
print(f"  • Range of affinities (mean): {ra_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {ra_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {ra_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats_ra.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

print("\n" + "=" * 70)

COHERENCY ANALYSIS SUMMARY

📊 PRP COHERENCY:
  • Mean CV (lower = more agreement): 0.606
  • Median CV: 0.569
  • Perfect agreement cases (CV=0): 0 / 300
  • High disagreement cases (CV>1): 25 / 300
  • Range of affinities (mean): 51.610
  • Most coherent abstract: 19.0 (mean CV: 0.258)
  • Least coherent abstract: 35.0 (mean CV: 1.021)

📊 RA COHERENCY:
  • Mean CV (lower = more agreement): 0.557
  • Median CV: 0.471
  • Perfect agreement cases (CV=0): 14 / 796
  • High disagreement cases (CV>1): 54 / 796
  • Range of affinities (mean): 17.028
  • Most coherent abstract: 17.0 (mean CV: 0.225)
  • Least coherent abstract: 11.0 (mean CV: 1.414)

🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):
  1. llama3.1-8b ↔ llama4-16x17b: 19.267
  2. llama4-16x17b ↔ ministral-3-14b-cloud: 20.000
  3. gemma4-31b-cloud ↔ ministral-3-14b-cloud: 20.263

🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):
  1. mixtral-8x7b ↔ qwen2.5-7b: 12.352
  2. gemma4-31b-cloud ↔ llama4-16x17b: 14

## Coherency by Fixed Affinity Levels (CV + MAD)

This section extends coherency analysis with fixed affinity bands and a robust companion metric:

- Low: 0-40
- Moderate: 41-70
- High: 71-100

Metrics per abstract-target unit across models:
- CV = std / mean (with epsilon)
- MAD = mean absolute deviation from cross-model mean

CV can be unstable when mean affinity is very low, so a low-mean caution flag is included.

In [41]:
# PRP coherency by fixed levels
prp_unit_coherency_lvl = compute_unit_coherency(prp_df, target_col='PRP_Name')
prp_level_summary = summarize_coherency_by_level(prp_unit_coherency_lvl)

print('PRP level-wise coherency summary (CV + MAD):')
display(prp_level_summary)

print('\nMost incoherent PRP units by CV:')
display(
    prp_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent PRP units by MAD:')
display(
    prp_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (CV)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (MAD)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

PRP level-wise coherency summary (CV + MAD):


,Affinity_Level,n_units,mean_cv,median_cv,p90_cv,mean_mad,median_mad,mean_range,pct_low_mean_flag,pct_singleton
0,Low,167,0.742749,0.713191,1.121947,14.332335,12.500000,45.988024,0.0,0.0
1,Moderate,112,0.483239,0.476620,0.646311,19.051190,18.611111,62.955357,0.0,0.0
2,High,21,0.178372,0.159415,0.318719,10.611111,10.000000,35.809524,0.0,0.0



Most incoherent PRP units by CV:


,Abstract_Index,PRP_Name,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
62,11,IBR,8.333333,1.594992,11.111111,30.0,6,False
200,34,IBR,6.666667,1.549193,8.888889,20.0,6,False
204,35,CROF,22.000000,1.369306,21.200000,75.0,5,False
206,35,IBR,8.000000,1.369306,9.600000,20.0,5,False
209,35,System Services,16.000000,1.351359,17.200000,50.0,5,False
140,24,IBR,32.500000,1.252689,33.333333,90.0,6,False
152,26,IBR,20.000000,1.224745,20.000000,60.0,6,False
8,2,IBR,20.000000,1.224745,20.000000,60.0,6,False
216,37,CROF,22.500000,1.223232,20.000000,75.0,6,False
50,9,IBR,28.333333,1.215813,28.333333,75.0,6,False



Most incoherent PRP units by MAD:


,Abstract_Index,PRP_Name,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
248,42,IBR,48.666667,0.864643,35.333333,92.0,6,False
98,17,IBR,44.166667,0.912474,34.166667,90.0,6,False
140,24,IBR,32.500000,1.252689,33.333333,90.0,6,False
146,25,IBR,42.500000,0.898287,32.500000,85.0,6,False
279,47,Planning,59.166667,0.624247,31.111111,80.0,6,False
104,18,IBR,45.833333,0.787925,30.555556,80.0,6,False
44,8,IBR,36.666667,0.927540,30.000000,75.0,6,False
171,29,Planning,48.333333,0.697541,28.333333,80.0,6,False
50,9,IBR,28.333333,1.215813,28.333333,75.0,6,False
266,45,IBR,38.333333,0.913664,28.333333,80.0,6,False


In [42]:
# RA coherency by fixed levels
ra_unit_coherency_lvl = compute_unit_coherency(ra_df, target_col='RA2025_ID')
ra_level_summary = summarize_coherency_by_level(ra_unit_coherency_lvl)

print('RA level-wise coherency summary (CV + MAD):')
display(ra_level_summary)

print('\nMost incoherent RA units by CV:')
display(
    ra_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent RA units by MAD:')
display(
    ra_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (CV)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (MAD)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

RA level-wise coherency summary (CV + MAD):


,Affinity_Level,n_units,mean_cv,median_cv,p90_cv,mean_mad,median_mad,mean_range,pct_low_mean_flag,pct_singleton
0,Low,560,0.646404,0.588465,1.414214,5.910134,4.375,14.748214,5.178571,46.071429
1,Moderate,175,0.394956,0.386598,0.597026,9.795670,10.000,25.891429,0.000000,33.714286
2,High,44,0.152440,0.127045,0.341115,3.936641,0.000,10.795455,0.000000,54.545455



Most incoherent RA units by CV:


,Abstract_Index,RA2025_ID,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
186,13,37,45.0,1.414214,45.0,90.0,2,False
746,48,10,37.5,1.414214,37.5,75.0,2,False
404,29,1,32.5,1.414214,32.5,65.0,2,False
743,48,7,22.5,1.414214,22.5,45.0,2,False
516,34,32,20.0,1.414214,20.0,40.0,2,False
759,49,17,20.0,1.414214,20.0,40.0,2,False
517,34,33,20.0,1.414214,20.0,40.0,2,False
745,48,9,17.5,1.414214,17.5,35.0,2,False
138,11,8,15.0,1.414214,15.0,30.0,2,False
515,34,31,15.0,1.414214,15.0,30.0,2,False



Most incoherent RA units by MAD:


,Abstract_Index,RA2025_ID,mean_affinity,cv,mad_affinity,range,n_models,cv_low_mean_flag
186,13,37,45.000000,1.414214,45.000000,90.0,2,False
746,48,10,37.500000,1.414214,37.500000,75.0,2,False
606,39,28,55.000000,0.899954,35.000000,70.0,2,False
111,8,37,52.500000,0.875466,32.500000,65.0,2,False
404,29,1,32.500000,1.414214,32.500000,65.0,2,False
47,4,31,52.500000,0.875466,32.500000,65.0,2,False
653,42,4,50.000000,0.848528,30.000000,60.0,2,False
652,42,3,45.000000,0.942809,30.000000,60.0,2,False
233,16,33,40.000000,1.060660,30.000000,60.0,2,False
722,46,16,59.000000,0.543960,27.200000,70.0,5,False


In [43]:
# Pairwise model disagreement by level (PRP)
prp_pairwise_lvl, prp_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(prp_df, target_col='PRP_Name')

if not prp_pairwise_lvl_summary.empty:
    print('PRP pairwise disagreement by level (highest mean difference):')
    display(prp_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = prp_pairwise_lvl_summary[prp_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'PRP Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No PRP pairwise level summary available.')

PRP pairwise disagreement by level (highest mean difference):


,Affinity_Level,Model_1,Model_2,mean_diff,median_diff,std_diff,n_pairs
8,High,llama3.1-8b,qwen2.5-7b,20.952381,10.0,20.893722,21
1,High,gemma4-31b-cloud,llama4-16x17b,20.095238,20.0,10.133631,21
9,High,llama4-16x17b,ministral-3-14b-cloud,17.619048,15.0,9.825865,21
0,High,gemma4-31b-cloud,llama3.1-8b,17.476190,10.0,18.795263,21
4,High,gemma4-31b-cloud,qwen2.5-7b,17.476190,10.0,18.675168,21
11,High,llama4-16x17b,qwen2.5-7b,16.904762,10.0,14.272017,21
5,High,llama3.1-8b,llama4-16x17b,15.952381,10.0,12.208506,21
14,High,mixtral-8x7b,qwen2.5-7b,15.952381,10.0,15.541802,21
3,High,gemma4-31b-cloud,mixtral-8x7b,15.809524,10.0,16.348147,21
6,High,llama3.1-8b,ministral-3-14b-cloud,15.000000,10.0,17.029386,21


In [44]:
# Pairwise model disagreement by level (RA)
ra_pairwise_lvl, ra_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(ra_df, target_col='RA2025_ID')

if not ra_pairwise_lvl_summary.empty:
    print('RA pairwise disagreement by level (highest mean difference):')
    display(ra_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = ra_pairwise_lvl_summary[ra_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'RA Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No RA pairwise level summary available.')

RA pairwise disagreement by level (highest mean difference):


,Affinity_Level,Model_1,Model_2,mean_diff,median_diff,std_diff,n_pairs
3,High,gemma4-31b-cloud,mixtral-8x7b,35.000000,35.0,NaN,1
11,High,ministral-3-14b-cloud,qwen2.5-7b,30.285714,19.0,25.031409,7
7,High,llama3.1-8b,qwen2.5-7b,27.166667,17.0,28.568631,6
10,High,ministral-3-14b-cloud,mixtral-8x7b,25.000000,25.0,14.142136,2
9,High,llama4-16x17b,qwen2.5-7b,22.166667,19.5,19.903936,6
6,High,llama3.1-8b,ministral-3-14b-cloud,17.777778,10.0,19.860625,9
4,High,gemma4-31b-cloud,qwen2.5-7b,17.300000,9.5,20.011386,10
5,High,llama3.1-8b,llama4-16x17b,16.153846,15.0,14.455945,13
0,High,gemma4-31b-cloud,llama3.1-8b,12.500000,5.0,15.000000,12
8,High,llama4-16x17b,ministral-3-14b-cloud,10.500000,7.5,9.264628,10


In [45]:
# Compact interpretation table
interpretation_summary = pd.DataFrame(
    [
        {
            'Dataset': 'PRP',
            'Mean_CV': prp_unit_coherency_lvl['cv'].mean(),
            'Median_CV': prp_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': prp_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': prp_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * prp_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(prp_unit_coherency_lvl),
        },
        {
            'Dataset': 'RA',
            'Mean_CV': ra_unit_coherency_lvl['cv'].mean(),
            'Median_CV': ra_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': ra_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': ra_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * ra_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(ra_unit_coherency_lvl),
        },
    ]
)

print('Global coherency summary (CV + MAD):')
display(interpretation_summary)

print('Interpretation notes:')
print('- Low CV + Low MAD: strong cross-model agreement')
print('- High CV + Low mean affinity: inspect MAD before concluding disagreement')
print('- High CV + High MAD: substantive disagreement across models')

Global coherency summary (CV + MAD):


,Dataset,Mean_CV,Median_CV,Mean_MAD,Median_MAD,Pct_Low_Mean_Flag,N_Units
0,PRP,0.606359,0.569091,15.833556,15.777778,0.000000,300
1,RA,0.557255,0.471405,6.671540,5.000000,3.643216,796


Interpretation notes:
- Low CV + Low MAD: strong cross-model agreement
- High CV + Low mean affinity: inspect MAD before concluding disagreement
- High CV + High MAD: substantive disagreement across models
